In [16]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import CRUD

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser2"
password = "Kappa"

# Connect to database via CRUD Module
db = CRUD(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([

    html.Center(html.B(html.H1('CS-340 Dashboard CClarke 2/28/2026'))),

    html.Center(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'width': '25%'}
        )
    ),

    html.Hr(),

    # Dropdown Filter
    html.Div([
        html.Label("Filter by Animal Type"),
        dcc.Dropdown(
            id='filter-type',
            options=[
                        {'label': 'Reset', 'value': 'RESET'},
                        {'label': 'Water Rescue', 'value': 'WATER'},
                        {'label': 'Mountain or Wilderness Rescue', 'value': 'MOUNTAIN'},
                        {'label': 'Disaster or Individual Tracking', 'value': 'DISASTER'}
            ],
            value='ALL',
            clearable=False
        )
    ]),

    html.Hr(),

    # Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '120px',
            'width': '120px',
            'maxWidth': '180px'
        }
    ),

    html.Br(),
    html.Hr(),

    # Graph and Map Side by Side
    html.Div(style={'display': 'flex'}, children=[

        html.Div(
            id='graph-id',
            style={'width': '50%'}
        ),

        html.Div(
            id='map-id',
            style={'width': '50%'}
        )

    ])
])
#############################################
# Interaction Between Components / Controller
#############################################


@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    if filter_type == 'WATER':
        query = {
            "breed": {"$regex": "Labrador|Chesapeake Bay|Newfoundland"},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'MOUNTAIN':
        query = {
            "breed": {"$regex": "German Shepherd|Alaskan Malamute|Siberian Husky|Rottweiler"},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'DISASTER':
        query = {
            "breed": {"$regex": "Doberman|German Shepherd|Golden Retriever|Bloodhound|Rottweiler"},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:  # RESET
        query = {}

    data = pd.DataFrame.from_records(db.read(query))

    if '_id' in data.columns:
        data.drop(columns=['_id'], inplace=True)

    return data.to_dict('records')


# Second Chart
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None:
        return

    dff = pd.DataFrame(viewData)

    # Count number of animals per breed
    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']

    fig = px.bar(
        breed_counts,
        x='breed',
        y='count',
        title='Number of Animals per Breed',
        labels={'breed': 'Breed', 'count': 'Count'}
    )

    fig.update_layout(
        xaxis_tickangle=-45,
        height=500
    )

    return [dcc.Graph(figure=fig)]


# Highlight selected column
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Update Map Based on Selected Row
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, selected_rows):

    if viewData is None:
        return

    dff = pd.DataFrame(viewData)

    if not selected_rows:
        row = 0
    else:
        row = selected_rows[0]

    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(),
                dl.Marker(
                    position=[dff.iloc[row]['location_lat'],
                              dff.iloc[row]['location_long']],
                    children=[
                        dl.Tooltip(dff.iloc[row]['breed']),
                        dl.Popup([
                            html.H3("Animal Name"),
                            html.P(dff.iloc[row]['name'])
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://shallowleopard-decadetoga-3000.codio.io/proxy/8050/
